In [1]:
import pandas as pd
import ast
from langchain_community.vectorstores import FAISS
import voyageai

In [2]:
df = pd.read_csv("final_recipes_data_clean.csv")
df.set_index("id", inplace=True)

vo = voyageai.Client()
vector_store = FAISS.load_local("faiss_index", None, allow_dangerous_deserialization=True)

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [3]:
fridge_inventory = ["chicken", "tomato", "onion", "garlic"]

In [ ]:
user_profile = {
    "diet": "low_carb",            # could be: low_carb, high_protein, vegetarian, vegan, keto
    "allergies": ["nuts", "soy"],  # exclude recipes with these ingredients
    "preferences": ["spicy", "quick"]  # boost recipes that match
}

In [12]:
def search_recipes(query, inventory, user_profile, k=10):
    query_vec = vo.embed([query], model="voyage-3.5").embeddings
    results = vector_store.similarity_search_by_vector(query_vec[0], k=k)

    # --- Preference keyword expansion ---
    preference_keywords = {
        "spicy": ["spicy", "chili", "pepper", "jalapeno", "hot sauce"],
        "quick": ["quick", "easy", "fast", "weeknight", "30-minute"],
        "healthy": ["healthy", "low-fat", "light", "diet"],
        "comfort": ["comfort", "hearty", "classic", "homestyle"],
    }

    personalized = []
    for r in results:
        rid = int(r.metadata["recipe_id"])
        if rid not in df.index:
            continue

        recipe = df.loc[rid]

        # --- Parse ingredients ---
        try:
            ingredients = ast.literal_eval(recipe.get("ingredients_cleaned", "[]"))
        except:
            ingredients = recipe.get("ingredients_cleaned", "")

        # --- Inventory Coverage ---
        if isinstance(ingredients, list) and len(ingredients) > 0:
            available = [item for item in ingredients if any(x.lower() in item.lower() for x in inventory)]
            coverage = len(available) / len(ingredients)
        else:
            coverage = 0

        # --- Allergy Check ---
        if any(allergen.lower() in str(ingredients).lower() for allergen in user_profile["allergies"]):
            continue

        # --- Preference Boost (with keyword expansion) ---
        boost = 0
        fields_to_check = [
            str(recipe.get("name", "")).lower(),
            str(recipe.get("ingredients", "")).lower(),
            str(recipe.get("ingredients_cleaned", "")).lower(),
            str(recipe.get("tags_merged", "")).lower(),
        ]

        for pref in user_profile["preferences"]:
            for kw in preference_keywords.get(pref, [pref]):  # fallback to exact pref if not in map
                if any(kw in field for field in fields_to_check):
                    boost += 0.1
                    break  # don’t double-count same pref

        # --- Diet Filtering ---
        if user_profile["diet"] == "vegetarian":
            if any(meat in str(ingredients).lower() for meat in ["chicken", "beef", "fish", "pork", "lamb"]):
                continue

        score = coverage + boost
        personalized.append((recipe, coverage, boost, score))

    # Sort by score
    personalized.sort(key=lambda x: x[3], reverse=True)
    return personalized


In [13]:
def display_personalized(recipes):
    data = []
    for recipe, coverage, boost, score in recipes:
        try:
            ingredients = ast.literal_eval(recipe.get("ingredients_cleaned", "[]"))
        except:
            ingredients = recipe.get("ingredients_cleaned", "")

        data.append({
            "Recipe Name": recipe.get("name", "Unknown"),
            "Servings": recipe.get("servings", "N/A"),
            "Coverage": f"{coverage:.0%}",
            "Preference Boost": f"{boost:.0%}",
            "Final Score": f"{score:.2f}",
            "Ingredients": ", ".join(ingredients) if isinstance(ingredients, list) else str(ingredients),
        })

    return pd.DataFrame(data)

In [14]:
query = "chicken dinner"
recipes = search_recipes(query, fridge_inventory, user_profile, k=10)
display_personalized(recipes)


,Recipe Name,Servings,Coverage,Preference Boost,Final Score,Ingredients
0,Vermouth Chicken,4.0,50%,20%,0.70,"8 chicken drumsticks, 5 large potatoes, 2 larg..."
1,Simple Chicken Skillet Dinner,4.0,50%,10%,0.60,"1 tablespoon butter, 4 boneless chicken breast..."
2,Sunday Dinner Chicken,4.0,33%,20%,0.53,"1 cup rice, uncooked, 1 broiler-fryer chicken,..."
3,Country-Style Chicken Dinner,4.0,43%,10%,0.53,"3 cups frozen hash brown potatoes, 1 (2 7/8 oz..."
4,Chicken Rice Dinner,4.0,31%,20%,0.51,"½ cup flour, 1 teaspoon salt, ½ teaspoon peppe..."
5,Classic Chicken and Dumplings,8.0,27%,20%,0.47,"1 (3 & ¼ lb) whole chickens, ½ teaspoon garlic..."
6,Farmhouse Chicken Dinner,4.0,30%,10%,0.40,"¼ cup flour, ½ teaspoon pepper, 4 small bone-i..."
7,Charishma's Delicious Chicken,4.0,19%,20%,0.39,"8 chicken thighs, washed and cleaned, 1 large ..."
8,Chicken Dinner in a Pot,4.0,29%,10%,0.39,"4 skinless chicken breast halves, 4 medium pot..."
9,Tim Mcgraw's Chicken and Dumplings,4.0,14%,10%,0.24,"1 small chicken, water, 1 pinch salt, 2 cups a..."
